# Derm ReasonEdit

`dermedit` kernel. Key is `standalone_dual` (DINOv2-large CLS + sbert, weight from `scalers.yaml`).
Cells above inspect one `(user, model)`. The last code cell runs all six and prints the table.

## 0. Switches — pick the user and the VLM

In [1]:
import sys
from pathlib import Path

# Locate editing/shared (config + data_engineering) and editing/reliability.
def _find_shared():
    hard = Path("/scratch/jq2uw/MME/derm_vlm_edit/editing/shared")
    for p in [Path.cwd(), Path.cwd().parent, hard, *Path.cwd().parents]:
        if (p / "config.py").exists() and p.name == "shared":
            return p
        cand = p / "editing" / "shared"
        if (cand / "config.py").exists():
            return cand
    return hard

SHARED = _find_shared()
RELIABILITY = SHARED.parent / "reliability"
assert (SHARED / "config.py").exists(), f"editing/shared/config.py not found (looked near {SHARED})"
sys.path.insert(0, str(SHARED))
sys.path.insert(0, str(RELIABILITY))
print("shared root:", SHARED)

import config as C
sys.path.insert(0, str(C.DERM_EDIT_ROOT))

# ---- experiment switches -------------------------------------------------
USER           = "jesse.y.liu.md@gmail.com"   # "jesse.y.liu.md@gmail.com" | "achiou@stanford.edu" | "lhunterellul"
MODEL_SHORT    = "medgemma"            # "medgemma" | "dermatollama"
RATIONALE_MODE = C.RATIONALE_MODE    # "edited_only" (human-authored) | "final"
NORM_ANSWER    = "y16"       # "y16" (class flip) | "human" (exact canonical text)
SUBSAMPLE      = 0                     # 0 = use all edits; else cap for a quick smoke run
MAX_NEW_TOKENS = C.MAX_NEW_TOKENS      # 512 — top-3 + reasoning, same as collection

# Standalone dual key. lang_scaler is read from auto_scaler/scalers.yaml (13.0).
OVERWRITE      = False               # True reruns even when the JSON is on disk
EMBED_MODE     = "standalone_dual"
IMAGE_ENCODER  = "dinov2"
IMAGE_POOL     = "cls"
TEXT_ENCODER   = "sbert"
SCALER_DATASET = "coco"

ANNOT_MODEL = C.MODEL_SHORT_TO_ANNOT[MODEL_SHORT]
print(f"user   = {USER}")
print(f"model  = {MODEL_SHORT}  (annotation label: {ANNOT_MODEL})")
print(f"rationale_mode = {RATIONALE_MODE} | norm_answer = {NORM_ANSWER} | subsample = {SUBSAMPLE} | max_new_tokens = {MAX_NEW_TOKENS}")
print(f"embed  = {EMBED_MODE} | {IMAGE_ENCODER}/{IMAGE_POOL} + {TEXT_ENCODER} | dataset = {SCALER_DATASET}")
print(f"annotations exists: {C.ANNOTATIONS_CSV.exists()} | images exists: {C.IMAGE_ROOT.exists()}")

shared root: /scratch/jq2uw/MME/derm_vlm_edit/editing/shared
user   = jesse.y.liu.md@gmail.com
model  = medgemma  (annotation label: MedGemma)
rationale_mode = edited_only | norm_answer = y16 | subsample = 0 | max_new_tokens = 512
embed  = standalone_dual | dinov2/cls + sbert | dataset = coco
annotations exists: True | images exists: True


## 1. Build the edit set (model's top-1 errors for this user)

In [2]:
import json
from data_engineering.build_edit_set import build as build_edit_set

edits, summary = build_edit_set(USER, ANNOT_MODEL, RATIONALE_MODE, write=True, norm_answer=NORM_ANSWER)
if SUBSAMPLE and len(edits) > SUBSAMPLE:
    edits = edits[:SUBSAMPLE]

assert edits, f"No usable edits for user={USER} model={ANNOT_MODEL}."
print(json.dumps({k: v for k, v in summary.items() if not k.endswith("_counts")}, indent=2))
print(f"\n{len(edits)} edits -> {C.edit_set_path(USER, ANNOT_MODEL)}")

{
  "user": "jesse.y.liu.md@gmail.com",
  "model": "MedGemma",
  "rationale_mode": "edited_only",
  "norm_answer": "y16",
  "question": "You are an expert dermatologist. Please give the top 3 diagnoses in your differential, and provide reasoning for each diagnosis, in format:\n1. [Diagnosis]: [Reasoning]\n2. [Diagnosis]: [Reasoning]\n3. [Diagnosis]: [Reasoning]",
  "fact_template": "New Fact: The top 1 most likely differential is {answer}.",
  "n_marked_incorrect": 25,
  "n_edits": 24,
  "n_dropped_same_grain": 1,
  "n_images_missing": 0,
  "n_multi_dx_top1": 0,
  "n_y16_unmapped": 0,
  "n_empty_rationale": 0
}

24 edits -> /sfs/weka/scratch/jq2uw/MME/derm_vlm_edit/editing/reliability/results/edit_set__jesse_y_liu_md_at_gmail_com__MedGemma.jsonl


## 2. Load the VLM + resolve ReasonEdit layer names
First model load downloads weights (can be slow / needs `HF_TOKEN` for gated repos).

In [3]:
from run_efficacy import result_path

RUN = OVERWRITE or not result_path(USER, MODEL_SHORT).exists()
if not RUN:
    print(f"result on disk, not loading {MODEL_SHORT}")
else:
    import torch
    sys.path.insert(0, str(C.DERM_EDIT_ROOT))
    from reasonedit import ReasonEdit, VQAModel, load_config
    from run_efficacy import is_correct, resolve_layers
    from derm_dataset import DermEditDataset

    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    config = load_config(
        MODEL_SHORT, device=device,
        batch_size=1, max_new_tokens=MAX_NEW_TOKENS, seed=42,
        mode=EMBED_MODE,
        image_encoder=IMAGE_ENCODER,
        image_encoder_pool=IMAGE_POOL,
        text_encoder=TEXT_ENCODER,
        scaler_dataset=SCALER_DATASET,
    )
    config.editor.fact_template = C.FACT_TEMPLATE
    model = VQAModel(config)
    resolve_layers(model, config)
    print("device:", device)
    print("fact_template:", C.FACT_TEMPLATE)


result on disk, not loading medgemma


### 2.5 Sanity — confirm the ReasonEdit vision hook layer resolves (GPU)
`dual_sbert` only hooks `inner_params_vision[0]`. `resolve_layers` (cell above) already mapped the
prefix-free yaml name to the exact live param (Mllama is a PeftModel, so names carry a
`base_model.model.` prefix). This cell verifies that resolution against the loaded model.

In [4]:
if not RUN:
    print('skipped, result on disk')
else:
    import re
    from reasonedit.utils import validate_and_correct_param_name

    params = dict(model.model.named_parameters())
    resolved = config.model.inner_params_vision[0]   # already corrected by resolve_layers
    print("inner_params_vision ->", resolved)
    print("exists in model     :", resolved in params,
          "| shape:", tuple(params[resolved].shape) if resolved in params else None)

    # show the tail of the vision-MLP weight list so you can pick a different late block if desired
    vis = sorted(n for n in params if re.search(r"vision.*mlp\.(fc1|fc2|up_proj|down_proj)\.weight$", n))
    print(f"\n{len(vis)} vision-mlp weight params; last 5:")
    for n in vis[-5:]:
        print("  ", n)


skipped, result on disk


## 3. Baseline top-3 generation (pre-edit; eval extracts top-1)

In [5]:
if not RUN:
    print('skipped, result on disk')
else:
    ds = DermEditDataset(config, edits)
    ds.set_dataloader()

    print("generating pre-edit top-3 (eval extracts top-1) ...")
    ds.task_generate(model, use_cache=True, max_new_tokens=MAX_NEW_TOKENS)
    pre = [ex.get("pred", {}).get("answer", "") for ex in ds.data]
    pre_raw = [ex.get("pred", {}).get("raw", "") for ex in ds.data]
    for ex, pa in list(zip(ds.data, pre))[:5]:
        print(f"  {ex['case_id']:>16}  pred={pa!r:40}  human={ex['answer']!r}")


skipped, result on disk


In [6]:
if not RUN:
    print('skipped, result on disk')
else:
    import torch
    from PIL import Image

    ex = ds.data[0]
    img = ex["image"]
    img = Image.open(img).convert("RGB") if isinstance(img, str) else img

    inputs = model.preprocess([img], [ex["prompt"]], model.processor)   # CPU tensors
    ids = inputs["input_ids"]
    emb = model.model.get_input_embeddings()
    cfg = model.model.config
    img_tok = getattr(cfg, "image_token_index", None) or getattr(cfg, "image_token_id", None)

    print("keys        :", list(inputs.keys()))
    print("input_ids   :", tuple(ids.shape), "min", int(ids.min()), "max", int(ids.max()))
    print("num_embeds  :", emb.num_embeddings)
    print("TOKEN OOB?  :", int(ids.max()) >= emb.num_embeddings)      # <-- the smoking gun
    if "pixel_values" in inputs:
        print("pixel_values:", tuple(inputs["pixel_values"].shape))
    if img_tok is not None:
        print("image_tok id:", img_tok, "count in ids:", int((ids == img_tok).sum()))


skipped, result on disk


## 4. Edit (build codebook) → apply retrieval → regenerate

In [7]:
if not RUN:
    print('skipped, result on disk')
else:
    print("building ReasonEdit codebook ...")
    editor = ReasonEdit(config, model)
    editor.edit(config, edit_ds=ds)
    editor.apply_to_dataset(ds)

    print("generating post-edit top-3 (eval extracts top-1) ...")
    ds.task_generate(model, use_cache=True, max_new_tokens=MAX_NEW_TOKENS)
    post = [ex.get("pred", {}).get("answer", "") for ex in ds.data]
    post_raw = [ex.get("pred", {}).get("raw", "") for ex in ds.data]

    retr_log = {r["uid"]: r.get("facts", []) for r in (editor.last_retrieval_log or [])}
    for ex, qa in list(zip(ds.data, post))[:5]:
        print(f"  {ex['case_id']:>16}  post={qa!r:40}  human={ex['answer']!r}")


skipped, result on disk


## 5. Reliability before vs after (the 0 → 1 flip)

In [8]:
if not RUN:
    print('skipped, result on disk')
else:
    rows = []
    n_pre = n_post = 0
    for ex, pa, qa, pa_raw, qa_raw in zip(ds.data, pre, post, pre_raw, post_raw):
        hn, hy = ex["answer"], ex.get("human_top1_y16")
        pc, qc = is_correct(pa, hn, hy, NORM_ANSWER), is_correct(qa, hn, hy, NORM_ANSWER)
        n_pre += int(pc); n_post += int(qc)
        rows.append({
            "uid": ex["uid"], "case_id": ex.get("case_id"),
            "model_top1": ex.get("model_top1"),
            "human_top1": ex.get("human_top1_raw"), "human_norm": hn, "human_y16": hy,
            "pred_pre": pa, "pred_post": qa,
            "pred_pre_raw": pa_raw, "pred_post_raw": qa_raw,
            "correct_pre": pc, "correct_post": qc,
            "retrieved_facts": retr_log.get(ex["uid"], []),
        })

    n = len(rows)
    n_pre = C.cap_before(n_pre)
    result = {
        "user": USER, "model": ANNOT_MODEL, "short_model": MODEL_SHORT,
        "rationale_mode": RATIONALE_MODE, "norm_answer": NORM_ANSWER, "question": C.QUESTION,
        "fact_template": C.FACT_TEMPLATE,
        "embed_mode": EMBED_MODE,
        "image_encoder": IMAGE_ENCODER,
        "image_pool": IMAGE_POOL,
        "text_encoder": TEXT_ENCODER,
        "scaler_dataset": SCALER_DATASET,
        "lang_scaler": float(editor.lang_scaler),
        "n_edits": n,
        "reliability_pre": n_pre / n if n else 0.0,
        "reliability_post": n_post / n if n else 0.0,
        "n_correct_pre": n_pre, "n_correct_post": n_post,
        "inner_params_vision": getattr(config.model, "inner_params_vision", None),
        "cases": rows,
    }
    print(f"reliability_pre  = {result['reliability_pre']:.3f} ({n_pre}/{n})")
    print(f"reliability_post = {result['reliability_post']:.3f} ({n_post}/{n})")
    print(f"flipped wrong->right: {sum(r['correct_post'] and not r['correct_pre'] for r in rows)}")


skipped, result on disk


## 6. Per-case table + before/after cards

In [9]:
if not RUN:
    print('skipped, result on disk')
else:
    for r in rows:
        flag = "OK " if r["correct_post"] else "-- "
        chg  = "  (flip)" if r["correct_post"] and not r["correct_pre"] else ""
        print(f"{flag}{r['case_id']:>16} | model:{str(r['model_top1'])[:22]:22} | "
              f"pre:{str(r['pred_pre'])[:22]:22} -> post:{str(r['pred_post'])[:22]:22} | "
              f"human:{r['human_top1']}{chg}")


skipped, result on disk


In [10]:
# import textwrap
# import matplotlib.pyplot as plt
# from PIL import Image

# def show_cards(edits, rows, n=6):
#     n = min(n, len(edits))
#     fig, axes = plt.subplots(n, 2, figsize=(14, 3.3 * n), gridspec_kw={"width_ratios": [1, 2.6]})
#     if n == 1:
#         axes = axes.reshape(1, 2)
#     for i, (e, r) in enumerate(zip(edits[:n], rows[:n])):
#         try:
#             axes[i, 0].imshow(Image.open(e["image"]).convert("RGB"))
#         except Exception as exc:
#             axes[i, 0].text(0.5, 0.5, f"no image\n{exc}", ha="center")
#         axes[i, 0].axis("off")
#         axes[i, 0].set_title(("OK " if r["correct_post"] else "-- ") + str(e["case_id"]), fontsize=9)
#         facts = "\n".join(f"  - {f}" for f in r["retrieved_facts"]) or "  (none)"
#         txt = (
#             f"MODEL top-1 (wrong): {r['model_top1']}\n"
#             f"PRE  pred          : {r['pred_pre']}\n"
#             f"POST pred          : {r['pred_post']}\n"
#             f"HUMAN top-1        : {r['human_top1']}  (y16: {r['human_y16']})\n\n"
#             f"retrieved facts:\n{facts}\n\n"
#             f"human reasoning:\n{textwrap.fill(e.get('rationale', ''), 92)}"
#         )
#         axes[i, 1].axis("off")
#         axes[i, 1].text(0.0, 1.0, txt, va="top", ha="left", fontsize=8.5, family="monospace")
#     plt.tight_layout(); plt.show()

# show_cards(edits, rows, n=6)

## 7. Save results JSON (same schema as `run_efficacy.py`)

In [11]:
if not RUN:
    print('skipped, result on disk')
else:
    import os

    os.makedirs(C.RESULTS_DIR, exist_ok=True)
    safe_user = USER.replace("@", "_at_").replace(".", "_")
    out = C.RESULTS_DIR / f"reliability_standalone__{safe_user}__{MODEL_SHORT}.json"
    with open(out, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)
    print("saved ->", out)


skipped, result on disk


## All six combos

Run the setup cell first, then this. One model load per VLM, then the table.

In [12]:
import config as C
from run_efficacy import result_path
import make_report as MR
from IPython.display import Markdown, display

users = ["jesse.y.liu.md@gmail.com", "achiou@stanford.edu", "lhunterellul"]
models = ["medgemma", "dermatollama"]

pending_by_model = {}
for model_short in models:
    pending = []
    for user in users:
        if result_path(user, model_short).exists() and not OVERWRITE:
            print(f"{user} / {model_short}: loaded")
        else:
            pending.append(user)
    if pending:
        pending_by_model[model_short] = pending

if pending_by_model:
    import gc, torch
    from reasonedit import VQAModel, load_config
    from run_efficacy import evaluate, resolve_layers
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    for model_short, pending in pending_by_model.items():
        config = load_config(
            model_short, device=device, batch_size=1,
            max_new_tokens=MAX_NEW_TOKENS, seed=42,
            mode=EMBED_MODE, image_encoder=IMAGE_ENCODER,
            image_encoder_pool=IMAGE_POOL, text_encoder=TEXT_ENCODER,
            scaler_dataset=SCALER_DATASET,
        )
        config.editor.fact_template = C.FACT_TEMPLATE
        vlm = VQAModel(config)
        resolve_layers(vlm, config)
        for user in pending:
            evaluate(user, model_short, vlm, config, max_new_tokens=MAX_NEW_TOKENS)
        del vlm
        gc.collect()
        torch.cuda.empty_cache()

results = MR.load_results(C.RESULTS_DIR, "reliability_standalone__*.json")
rows, pooled = MR.build_rows(results, users, MR.VLMS)
display(Markdown(MR.render_markdown(rows, pooled, MR.VLMS)))


jesse.y.liu.md@gmail.com / medgemma: loaded
achiou@stanford.edu / medgemma: loaded
lhunterellul / medgemma: loaded
jesse.y.liu.md@gmail.com / dermatollama: loaded
achiou@stanford.edu / dermatollama: loaded
lhunterellul / dermatollama: loaded


| User | MedGemma Before | MedGemma After | MedGemma Δ | DermatoLlama Before | DermatoLlama After | DermatoLlama Δ |
|---|---|---|---|---|---|---|
| jesse.y.liu.md | 0% (0/24) | 100% (24/24) | +100 | 0% (0/14) | 100% (14/14) | +100 |
| achiou | 0% (0/20) | 100% (20/20) | +100 | 0% (0/13) | 100% (13/13) | +100 |
| lhunterellul | 0% (0/8) | 100% (8/8) | +100 | 33% (2/6) | 100% (6/6) | +67 |
| **Pooled** | 0% (0/52) | 100% (52/52) | +100 | 6% (2/33) | 100% (33/33) | +94 |
